In [ ]:
import pandas as pd
import os
import numpy as np

# Define a local demo data directory
base_dir = os.getcwd()  # current notebook folder
data_path = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "bank_customers"))

os.makedirs(data_path, exist_ok=True)

# Create a small synthetic banking dataset for demo purposes
pdf = pd.DataFrame({
    "customer_id": range(1001, 1011),
    "name": [
        "Sofia",
        "Lukas",
        "Amina",
        "Omar",
        "Kwame",
        "Zuri",
        "Hiroshi",
        "Mei",
        "Arun",
        "Sakura"
    ],
    "age": [25, 34, 29, 41, 32, 27, 45, 23, 31, 28],
    "account_type": [
        "checking","savings","checking","checking","savings",
        "savings","checking","checking","savings","checking"
    ],
    "balance": [
        5200.50, 11200.10, 3400.20, 21800.00, 670.00,
        44550.25, 7800.00, 1500.00, 9400.75, 21000.90
    ],
    "is_active": [True, True, False, True, False, True, True, False, True, True],
    "risk_score": np.random.randint(1, 10, size=10)  # synthetic risk metric
})

# Persist dataset in CSV format for reproducibility
csv_path = os.path.join(data_path, "bank_customers.csv")
pdf.to_csv(csv_path, index=False)

print("CSV created:", csv_path)


CSV created: \..\..\data\bank_customers\bank_customers.csv


In [2]:
import findspark
findspark.init()
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("read-bank-customers-csv")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

base_dir = os.getcwd()  # current notebook folder
csv_path = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "bank_customers"))

# Read CSV with Spark (header + schema inference)
df = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(csv_path)
)

print("Spark UI:", spark.sparkContext.uiWebUrl)
df.show()



Spark UI: http://host.docker.internal:4040
+-----------+-------+---+------------+--------+---------+----------+
|customer_id|   name|age|account_type| balance|is_active|risk_score|
+-----------+-------+---+------------+--------+---------+----------+
|       1001|  Sofia| 25|    checking|  5200.5|     true|         4|
|       1002|  Lukas| 34|     savings| 11200.1|     true|         6|
|       1003|  Amina| 29|    checking|  3400.2|    false|         3|
|       1004|   Omar| 41|    checking| 21800.0|     true|         5|
|       1005|  Kwame| 32|     savings|   670.0|    false|         6|
|       1006|   Zuri| 27|     savings|44550.25|     true|         4|
|       1007|Hiroshi| 45|    checking|  7800.0|     true|         7|
|       1008|    Mei| 23|    checking|  1500.0|    false|         1|
|       1009|   Arun| 31|     savings| 9400.75|     true|         3|
|       1010| Sakura| 28|    checking| 21000.9|     true|         6|
+-----------+-------+---+------------+--------+---------+---

In [3]:
pdf.head()

,customer_id,name,age,account_type,balance,is_active,risk_score
0,1001,Sofia,25,checking,5200.5,True,8
1,1002,Lukas,34,savings,11200.1,True,2
2,1003,Amina,29,checking,3400.2,False,6
3,1004,Omar,41,checking,21800.0,True,4
4,1005,Kwame,32,savings,670.0,False,3


In [ ]:
df2 = spark.createDataFrame(pdf)
df2.show()


+-----------+-------+---+------------+--------+---------+----------+
|customer_id|   name|age|account_type| balance|is_active|risk_score|
+-----------+-------+---+------------+--------+---------+----------+
|       1001|  Sofia| 25|    checking|  5200.5|     true|         8|
|       1002|  Lukas| 34|     savings| 11200.1|     true|         2|
|       1003|  Amina| 29|    checking|  3400.2|    false|         6|
|       1004|   Omar| 41|    checking| 21800.0|     true|         4|
|       1005|  Kwame| 32|     savings|   670.0|    false|         3|
|       1006|   Zuri| 27|     savings|44550.25|     true|         2|
|       1007|Hiroshi| 45|    checking|  7800.0|     true|         9|
|       1008|    Mei| 23|    checking|  1500.0|    false|         1|
|       1009|   Arun| 31|     savings| 9400.75|     true|         9|
|       1010| Sakura| 28|    checking| 21000.9|     true|         7|
+-----------+-------+---+------------+--------+---------+----------+



In [9]:
import os

base_dir = os.getcwd()  # current notebook folder
parquet_path = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "bank_customers_parquet"))

df2.write.mode("overwrite").parquet(parquet_path)
print("Written to:", parquet_path)

Written to: c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\data\bank_customers_parquet
